Juego de Poker


In [ ]:
import random
import numpy as np
#from tqdm import tqdm


In [ ]:

class PokerEnv:
    def __init__(self):
        self.reset()

    def reset(self):
        # Genera las cartas del jugador y de la mesa
        self.players_cart = np.zeros((2,3))
        self.table_carts = np.zeros((4,3))
        self.round = 0
        
        return self

    def asig_cart(self, n=1, PorT=True):
        # valores de A,1-10,J,Q,K
        valores = list(range(1, 13))
        # color rojo y negro
        colores = [0, 1]
        # figuras corazon, trevol, diamante, pica
        figura = [0,1,2,3]
        
        if(PorT):
            for i in range(n):
                for j in range(n):
                    if j==0:
                        self.table_carts.append(random.choice(valores))
                    if j==1:
                        self.table_carts.append(random.choice(colores))
                    if j==2:
                        self.table_carts.append(random.choice(figura))
            return self
        else:
            for i in range(n):
                for j in range(n):
                    if j==0:
                        self.players_cart.append(random.choice(valores))
                    if j==1:
                        self.players_cart.append(random.choice(colores))
                    if j==2:
                        self.players_cart.append(random.choice(figura))
            return self
        
    #Combinaciones de cartas
    def mismo_color(self):
        fullcolor = []
        fullcolor = fullcolor.append(self.table_carts)
        fullcolor = fullcolor.append(self.players_cart)

        #Por lo menos que sean 5 de los 6
        if fullcolor[:1] == 0 or fullcolor[:1] == 1:
            return True
        else:
            return False

    def misma_figura(self):
        fullfigure = []
        fullfigure.append(self.table_carts)
        fullfigure.append(self.players_cart)

        #Por lo menos que sean 5 de los 6
        if fullfigure[:2] == 0 or fullfigure[:2] == 1 or fullfigure[:2] == 3 or fullfigure[:2] == 4:
            return True
        else:
            return False

    def mismo_valor(self):
        fullvalor = []
        fullvalor.append(self.table_carts)
        fullvalor.append(self.players_cart)
        n_repetidos = []
        doble_repetido = []

        #Verifica cuantos numeros repetidos hay
        for i in range(fullvalor):
            if i == 0:
                n_repetidos.append(fullvalor[i])
            if n_repetidos[i] == fullvalor[i]:
                n_repetidos.append(fullvalor[i])
            else:
                if i < 5:
                    if doble_repetido[i] == fullvalor[i]:
                        doble_repetido.append(fullvalor[i])
                    elif doble_repetido[i] == fullvalor[i+1]:
                        doble_repetido.append(fullvalor[i+1])
        
        #Poker
        if n_repetidos == 4:
            return 5
        #Full
        elif n_repetidos == 3 & doble_repetido == 2:
            return 4
        #Trica
        elif n_repetidos == 3:
            return 3
        #Doble par
        elif n_repetidos == 2 & doble_repetido == 2:
            return 2
        #Par
        elif n_repetidos == 2:
            return 1
        #Carta Mayor
        elif n_repetidos == 1:
            return 0

    def escalera(self):
        EscarleraRealNormal = []
        EscarleraRealNormal.append(self.table_carts)
        EscarleraRealNormal.append(self.players_cart)
        VerificarEscaleraNormal = []
        VerificarEscaleraReal=[]        
        for i in range (EscarleraRealNormal):
            if i == 0:
                VerificarEscaleraNormal.append(EscarleraRealNormal[i])
            if VerificarEscaleraNormal[i]+1 == EscarleraRealNormal[i] or VerificarEscaleraNormal[i]-1 == EscarleraRealNormal[i]:
                if VerificarEscaleraNormal[i] == 1 or VerificarEscaleraNormal[i] == 10 or VerificarEscaleraNormal[i] == 11 or VerificarEscaleraNormal[i] == 12 or VerificarEscaleraNormal[i] == 13:
                    VerificarEscaleraReal.append(EscarleraRealNormal[i])
                VerificarEscaleraNormal.append(EscarleraRealNormal[i])                 

        #Verificar Si es Escalera real de A,10,J,Q,K
        if VerificarEscaleraReal == 5:
            return 3
        #Verificar si es Escalera normal de cualquier numero de forma escalada de solo 5
        elif VerificarEscaleraNormal == 5:
            return 2
        #No es ninguno
        else:
            return 1

    def steps(self):
        # acción = 0 -> retirarse, 1 -> continuar
        self.players_cart.append(self.asig_cart(self, n=2, PorT=False))
        self.round += 1
        if self.round <= 4:
            self.table_cards.append(self)
            done = False
        else:
            done = True
        return self, done

In [ ]:
class Game():
    def __init__(self, player1, player2):
        self.players = [player1, player2]
        self.board = PokerEnv()

    def selfplay(self, rounds=100):
        wins = [0, 0]
        for i in range(1, rounds + 1):
            self.board.reset()
            for player in self.players:
                player.reset()
            game_over = False
            while not game_over:
                for player in self.players:
                    action = player.move(self.board)
                    self.board.update(player.symbol, action[0], action[1])
                    for player in self.players:
                        player.update(self.board)
                    if self.board.is_game_over() is not None:
                        game_over = True
                        break
            self.reward()
            for ix, player in enumerate(self.players):
                if self.board.is_game_over() == player.symbol:
                    wins[ix] += 1
        return wins


    def reward(self):
        mejor_mazo = []
        mejor_mazo = mejor_mazo.append(self.board.table_carts)
        mejor_mazo = mejor_mazo.append(self.board.players_cart)

        winner = 0

        if self.board.mismo_color(self) == True:
            if self.board.escalera(self) == 3:
                winner = 10
            elif self.board.escalera(self) == 2:
                winner = 9
            elif self.board.misma_figura(self) ==True:
                winner = 5
        elif self.board.mismo_valor(self) == 5:
            winner = 8
        elif self.board.mismo_valor(self) == 4:
            winner = 7
        elif self.board.mismo_valor(self) == 3:
            winner = 6
        elif self.board.mismo_valor(self) == 2:
            winner = 4
        elif self.board.mismo_valor(self) == 1:
            winner = 3
